## Define libraries

In [ ]:
import sys
import os


# Get the absolute path to the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the parent directory to sys.path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [ ]:
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

# from helper import RAGHelper
from dotenv import load_dotenv
load_dotenv()
import os

## Define Variables

In [29]:
# Constants

DATASET_SOURCE = 'suniltvl/ragbench'
# DATASET_NAME = {"delucionqa":"Jeep manual", "emanual": "TV manual", "techqa":"Technotes"}
DATA_SPLIT = 'test'
VECTOR_DATABASES = ['chroma'] # ['chroma', 'milvus']
EMBEDDING_MODELS = ["BAAI/LLM-Embedder", "BAAI/bge-large-en-v1.5"]
MAX_CHUNKS = 5000
CHUNKING_SIZES = [256, 512, 1024]
CHUNKING_OVERLAPS = [50, 100, 200]
SEPARATORS = ["\n\n", "\n", " ", ".", ","]
DOMAINS = {
    'cs':
    {
        "delucionqa":"Jeep manual", 
        "emanual": "TV manual", 
        "techqa":"Technotes"
    },
    'gk':
    {
        "hotpotqa":"wiki 1",
        "msmacro":"web pages",
        "hagrid":"wiki 2",
        "expertqa":"googlesearch"
    }
}

### Templates

In [ ]:
from pathlib import Path

# ------------------------------------------------------------------
# Path Templates
# ------------------------------------------------------------------

def get_db_folder(db_type: str) -> Path:
    return Path(f"../database_{db_type}")


def get_collection_name(
    embedding_model: str,
    domain_name_key: str,
) -> str:
    return f"{embedding_model.replace('/', '_')}-{domain_name_key}"


def get_persist_directory(
    db_type: str,
    domain_name_key: str,
    chunk_size: int,
    chunk_overlap: int,
) -> Path:
    return (
        get_db_folder(db_type)
        / f"{domain_name_key}_{chunk_size}_{chunk_overlap}"
    )


### All Variables

In [ ]:
# all_db_folders = [get_db_folder(vdb) for vdb in EMBEDDING_MODELS]

# all_collections = []
# for db_folder in EMBEDDING_MODELS:
#     all_collections.extend(list(db_folder.iterdir()))

# all_collections

## Helper Functions

In [ ]:
def deduplicate_data(data, doc_type):
    data_dict = {}
    for d in data:
        document = " ".join(d["documents"])
        if document in data_dict:
            data_dict[document]["docid"].append(d["id"])
        else:
            data_dict[document] = {"docid":[d["id"]]}
            
    for k in data_dict:
        data_dict[k]["document_type"] = doc_type
        
    return data_dict

### Data Fetching

In [ ]:
def get_docs(db_name: str):
    dataset = load_dataset(DATASET_SOURCE, db_name, split=DATA_SPLIT)
    dedup = deduplicate_data(dataset, db_name)
    docs = [
        Document(
            metadata=metadata, 
            page_content=content
        )
        for content, metadata in dedup.items()
    ]   
    return docs


### Chunking

In [ ]:
def chunk_documents(docs: list,chunk_size: int, chunk_overlap: int):    

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, separators=SEPARATORS)
    docs_chunks = text_splitter.split_documents(docs)
    
    return docs_chunks

### Embedding Models

In [ ]:
def get_embedding_function(model: str):
    return HuggingFaceEmbeddings(model=model)

### Vectorization

In [ ]:
def vectorize_documents(docs_chunks: list, db_name: str, persist_directory: str, collection_name: str, embedding_function):

    if len(docs_chunks) > MAX_CHUNKS:
        for i in range(0, len(docs_chunks), MAX_CHUNKS):
            doc_chunk = docs_chunks[i:i+MAX_CHUNKS]
    
            if os.path.exists(persist_directory) and os.listdir(persist_directory):
                print(f"Loading existing vector database...{db_name}")
                vector_db = Chroma(
                    collection_name=collection_name,
                    persist_directory=persist_directory, 
                    embedding_function=embedding_function
                )
                vector_db.add_documents(doc_chunk)
            else:
                print(f"Creating new vector database...{db_name}")
                vector_db = Chroma.from_documents(
                    collection_name=collection_name,
                    documents=doc_chunk, 
                    embedding=embedding_function, 
                    persist_directory=persist_directory
                )
    else:
        # FIXED: This 'if' statement is now perfectly aligned with the 'else' below it
        if os.path.exists(persist_directory) and os.listdir(persist_directory):
            print(f"Loading existing vector database...{db_name}")
            vector_db = Chroma(
                collection_name=collection_name,
                persist_directory=persist_directory, 
                embedding_function=embedding_function
            )
            vector_db.add_documents(docs_chunks)
        else:
            print(f"Creating new vector database...{db_name}")
            vector_db = Chroma.from_documents(
                collection_name=collection_name,
                documents=docs_chunks, 
                embedding=embedding_function, 
                persist_directory=persist_directory
            )
    return vector_db

## Looping Domains

In [ ]:
def loop_domains():   
    # Loop through domains and datasets
    for domain_short_name, data_set in DOMAINS.items():

        print(f"Processing domain: {domain_short_name}")        
        # Loop through datasets
        for data_set_path, name in data_set.items():

            print(f"  Processing dataset: {data_set_path} > {name}")
            
            # Read from dataset (HuggingFace)
            docs = get_docs(data_set_path)

            print(f"  Number of documents: {len(docs)} in {data_set_path}")

            # Loop through chunking sizes and overlaps
            for chunk_size, chunk_overlap in zip(CHUNKING_SIZES, CHUNKING_OVERLAPS):

                print(f"   Processing chunk size: {chunk_size}, overlap: {chunk_overlap}")

                # Chunk documents
                chunks = chunk_documents(docs, chunk_size, chunk_overlap)

                print(f"   Number of chunks: {len(chunks)} in {data_set_path} for chunk size: {chunk_size}, overlap: {chunk_overlap}")

                # Loop Embedder function
                for embedder in EMBEDDING_MODELS:
                    print(f"      Processing embedder: {embedder}")
                    # Generate embeddings
                    embedding_function = get_embedding_function(embedder)

                    # Loop Vector Database
                    for vector_db_type in VECTOR_DATABASES:
                        print(f"         Processing vector database: {vector_db_type} for embedder: {embedder}, chunk size: {chunk_size}, overlap: {chunk_overlap}, data set: {data_set_path}, domain: {domain_short_name}")

                        # Get DB Name
                        db_name = get_db_folder(vector_db_type)
                        
                        # Get Collection Name
                        collection_name = get_collection_name(embedder, domain_short_name)

                        # Get persist directory
                        persist_directory = get_persist_directory(vector_db_type, domain_short_name, chunk_size, chunk_overlap)                    

                        print(f"         Persist directory: {persist_directory}, Collection name: {collection_name}, db_name: {db_name}")
                        vector_db = vectorize_documents(chunks, db_name, persist_directory, collection_name, embedding_function)

                    


In [30]:
loop_domains()

Processing domain: cs
  Processing dataset: delucionqa > Jeep manual
  Number of documents: 92 in delucionqa
   Processing chunk size: 256, overlap: 50
   Number of chunks: 2024 in delucionqa for chunk size: 256, overlap: 50
      Processing embedder: BAAI/LLM-Embedder


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

         Processing vector database: chroma for embedder: BAAI/LLM-Embedder, chunk size: 256, overlap: 50, data set: delucionqa, domain: cs
         Persist directory: ..\database_chroma\cs_256_50, Collection name: BAAI_LLM-Embedder-cs, db_name: ..\database_chroma
Loading existing vector database.....\database_chroma
      Processing embedder: BAAI/bge-large-en-v1.5


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

f:\Github\Capstone\nextgen-rag-system\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sunil\.cache\huggingface\hub\models--BAAI--bge-large-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


...

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

         Processing vector database: chroma for embedder: BAAI/bge-large-en-v1.5, chunk size: 256, overlap: 50, data set: delucionqa, domain: cs
         Persist directory: ..\database_chroma\cs_256_50, Collection name: BAAI_bge-large-en-v1.5-cs, db_name: ..\database_chroma
Loading existing vector database.....\database_chroma
   Processing chunk size: 512, overlap: 100
   Number of chunks: 1026 in delucionqa for chunk size: 512, overlap: 100
      Processing embedder: BAAI/LLM-Embedder


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

         Processing vector database: chroma for embedder: BAAI/LLM-Embedder, chunk size: 512, overlap: 100, data set: delucionqa, domain: cs
         Persist directory: ..\database_chroma\cs_512_100, Collection name: BAAI_LLM-Embedder-cs, db_name: ..\database_chroma
Creating new vector database.....\database_chroma


KeyboardInterrupt: 

In [ ]:
for k, v in DATASET_NAME.items():
    print(f"Creating embeddings for {k} using {embedding_model}")
    MAX_CHUNKS = 5000

    dataset = load_dataset(DATASET_SOURCE, k, split=DATA_SPLIT)
    dedup = deduplicate_data(dataset, v)
    docs = [
        Document(
            metadata=v, 
            page_content=k
        )
        for k, v in dedup.items()
    ]
    
    for cs, co in zip(CHUNKING_SIZES, CHUNKING_OVERLAPS):
        db_name = f"{DOMAIN}_{cs}_{co}"
        persist_directory = f"{chromadb_folder}/{db_name}"
            
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=cs, chunk_overlap=co, separators=SEPARATORS)
        docs_chunks = text_splitter.split_documents(docs)
        print(len(docs_chunks))
        
        if len(docs_chunks) > MAX_CHUNKS:
            for i in range(0, len(docs_chunks), MAX_CHUNKS):
                doc_chunk = docs_chunks[i:i+MAX_CHUNKS]
        
                if os.path.exists(persist_directory) and os.listdir(persist_directory):
                    print(f"Loading existing vector database...{db_name}")
                    vector_db = Chroma(
                        collection_name=collection_name,
                        persist_directory=persist_directory, 
                        embedding_function=embedding_function
                    )
                    vector_db.add_documents(doc_chunk)
                else:
                    print(f"Creating new vector database...{db_name}")
                    vector_db = Chroma.from_documents(
                        collection_name=collection_name,
                        documents=doc_chunk, 
                        embedding=embedding_function, 
                        persist_directory=persist_directory
                    )
        else:
            # FIXED: This 'if' statement is now perfectly aligned with the 'else' below it
            if os.path.exists(persist_directory) and os.listdir(persist_directory):
                print(f"Loading existing vector database...{db_name}")
                vector_db = Chroma(
                    collection_name=collection_name,
                    persist_directory=persist_directory, 
                    embedding_function=embedding_function
                )
                vector_db.add_documents(docs_chunks)
            else:
                print(f"Creating new vector database...{db_name}")
                vector_db = Chroma.from_documents(
                    collection_name=collection_name,
                    documents=docs_chunks, 
                    embedding=embedding_function, 
                    persist_directory=persist_directory
                )

print(f"Data load into vector database completed for {embedding_model}")